# Guardrails RLHF — Colab (TRL GRPO)

Use after exporting `grpo_train.jsonl` from the repo (`python -m rlhf.pipeline.export_grpo_dataset`).
Install GPU stack in Colab only; do not add torch/trl to the main app requirements.

In [ ]:
# §1 GPU check
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Select GPU runtime (e.g. A100)"

In [ ]:
# §2 Install (Colab) — versions are indicative; pin to your Colab image
%pip install -q "torch>=2.4" "transformers>=4.44" "trl>=0.12" "peft>=0.12" accelerate datasets

In [ ]:
# §3 Mount Drive (optional) + clone / sync repo branch
# from google.colab import drive
# drive.mount('/content/drive')
# !git clone -b deploy https://github.com/shashidharbabu/guardrails-enterprise.git /content/repo

import sys, shutil, pathlib
REPO = pathlib.Path('/content/repo')   # adjust if cloned elsewhere
sys.path.insert(0, str(REPO))

# Copy the standalone reward helper so Colab can find it without the full package tree
dest = pathlib.Path('/content/reward_fn')
dest.mkdir(exist_ok=True)
shutil.copy(REPO / 'openrlhf/reward_fn/healthcare_reward.py', dest / 'healthcare_reward.py')
(dest / '__init__.py').touch()
print('reward_fn copied to', dest)


In [ ]:
# §4 Load JSONL export and build HF Dataset with Qwen chat-template prompts
from datasets import Dataset
from transformers import AutoTokenizer
import json, sys

JSONL_PATH = '/content/grpo_train.jsonl'  # output of: python -m rlhf.pipeline.export_grpo_dataset
MODEL_ID   = 'Qwen/Qwen2.5-7B-Instruct'

SYSTEM_PROMPT = (
    'You are Agent A in a Multi-Agent Debate pipeline for healthcare regulatory AI. '
    'Your role is to provide a well-reasoned, evidence-based claim with a calibrated confidence score.\n'
    'RULES:\n'
    '(1) Never include PHI or patient identifiers.\n'
    '(2) Always cite a specific source: HIPAA section, CFR part, drug label, or peer-reviewed study.\n'
    '(3) End your response with: CONFIDENCE: <float 0.0-1.0>.\n'
    '(4) If uncertain, express lower confidence — do not hallucinate certainty.'
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

rows = []
with open(JSONL_PATH) as f:
    for line in f:
        r = json.loads(line)
        rows.append(r)

def make_prompt(query: str) -> str:
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": query},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

ds = Dataset.from_list([
    {
        'prompt':       make_prompt(r['prompt']),
        'judge_verdict': float(r.get('judge_verdict', 0.5)),
    }
    for r in rows
])
print(f'Dataset: {len(ds)} examples loaded')
print('Sample prompt (first 300 chars):', ds[0]['prompt'][:300])


## §5 Train with TRL GRPOTrainer

Runs GRPO / REINFORCE++ on Qwen2.5-7B-Instruct + LoRA.  
Reward = Brier calibration score + citation bonus + judge bonus − PHI penalty.  
See [docs/RLHF_IMPLEMENTATION.md](../docs/RLHF_IMPLEMENTATION.md) for full derivation.


In [ ]:
# §5 Load model + LoRA, configure GRPO, run training
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from trl import GRPOConfig, GRPOTrainer
from reward_fn.healthcare_reward import scalar_reward

# ── Model ──────────────────────────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)

# ── LoRA ───────────────────────────────────────────────────────────────────
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()  # ~40M / 7B ≈ 0.57%

# ── Reward function (wraps scalar_reward for TRL batch hook) ───────────────
def grpo_reward_fn(completions, **kwargs):
    """TRL passes completions as a list of strings; judge verdicts come from the dataset."""
    judge_verdicts = kwargs.get('judge_verdict', [0.5] * len(completions))
    return scalar_reward.__module__ and [
        scalar_reward(c, float(v))
        for c, v in zip(completions, judge_verdicts)
    ]

# ── GRPOConfig — matches RLHF_IMPLEMENTATION.md §8 ────────────────────────
training_args = GRPOConfig(
    output_dir='/content/guardrails-grpo',
    # Training duration
    max_steps=150,                     # ~22 Colab compute units on A100
    # Batch
    per_device_train_batch_size=1,     # A100 40 GB w/ 7B model
    gradient_accumulation_steps=4,     # effective batch = 4
    num_generations=2,                 # REINFORCE++ group size (memory constraint)
    max_completion_length=256,         # Agent A response cap
    # Optimiser
    learning_rate=5e-6,
    lr_scheduler_type='cosine',
    warmup_steps=10,
    # RL
    beta=0.05,                         # KL penalty — allow meaningful policy change
    # Precision
    bf16=True,                         # A100 native bfloat16
    gradient_checkpointing=True,       # reduces activation memory
    # Logging
    logging_steps=10,
    save_steps=50,
    report_to='none',                  # set to 'wandb' to enable W&B tracking
)

# ── Trainer ────────────────────────────────────────────────────────────────
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=ds,
    reward_funcs=[grpo_reward_fn],
    processing_class=tokenizer,
)

# ── Train ──────────────────────────────────────────────────────────────────
trainer.train()
trainer.save_model('/content/guardrails-grpo/final')
print('Training complete. Model saved to /content/guardrails-grpo/final')


In [ ]:
# §6 Quick eval — compare reward before / after training
from reward_fn.healthcare_reward import scalar_reward, extract_confidence, has_citation

EVAL_PROMPTS = [
    'What are the HIPAA technical safeguard requirements under 45 CFR §164.312?',
    'When must an adverse event be reported to the FDA under 21 CFR §803?',
]

model.eval()
for prompt in EVAL_PROMPTS:
    inputs = tokenizer(make_prompt(prompt), return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    completion = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    conf = extract_confidence(completion)
    cited = has_citation(completion)
    reward = scalar_reward(completion, judge_verdict=0.8)  # assume 0.8 judge verdict for eval
    print(f'Prompt: {prompt[:60]}...')
    print(f'  confidence={conf}  cited={cited}  reward={reward:.4f}')
    print(f'  completion[:200]: {completion[:200]}\n')
